# 01. Keşifsel Veri Analizi (Exploratory Data Analysis - EDA)
### LEVIR-CD Uydu Görüntüleri ile Değişim Tespiti (Change Detection)

Bu notebook projedeki veri setini incelemek, bitemporal (T1 ve T2) çiftlerin özelliklerini analiz etmek ve sınıf dengesizliğini (class imbalance) ölçmek için hazırlanmıştır.

In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Ayarlar
RAW_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'
print('Raw Dir exists:', os.path.exists(RAW_DIR))
print('Processed Dir exists:', os.path.exists(PROCESSED_DIR))

## 1. Veri Seti İstatistikleri & Sınıf Dağılımı Analizi
Change detection problemlerinde değişim pikselleri genellikle tüm görüntünün %3-5'ini oluşturur. Bu dengesizlik Dice Loss ve Focal Loss kullanımını zorunlu kılar.

In [ ]:
def analyze_split(split='train'):
    lbl_paths = glob.glob(os.path.join(PROCESSED_DIR, split, 'label', '*.*'))
    total_pixels = 0
    change_pixels = 0
    
    for p in lbl_paths:
        mask = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            change_pixels += np.sum(mask > 0)
            total_pixels += mask.size
            
    ratio = (change_pixels / total_pixels * 100) if total_pixels > 0 else 0
    print(f'[{split.upper()}] Toplam yama: {len(lbl_paths)}')
    print(f'[{split.upper()}] Toplam piksel: {total_pixels:,}')
    print(f'[{split.upper()}] Değişen piksel: {change_pixels:,} ({ratio:.2f}%)')
    return ratio

# train ve val splitlerini analiz et
for s in ['train', 'val', 'test']:
    if os.path.exists(os.path.join(PROCESSED_DIR, s)):
        analyze_split(s)

## 2. Bitemporal Çiftleri ve Etiketleri Görselleştirme

In [ ]:
sample_files = glob.glob(os.path.join(PROCESSED_DIR, 'train', 'A', '*.*'))[:3]

for sf in sample_files:
    name = os.path.basename(sf)
    t1 = cv2.cvtColor(cv2.imread(sf), cv2.COLOR_BGR2RGB)
    t2 = cv2.cvtColor(cv2.imread(os.path.join(PROCESSED_DIR, 'train', 'B', name)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(os.path.join(PROCESSED_DIR, 'train', 'label', name), cv2.IMREAD_GRAYSCALE)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(t1)
    axes[0].set_title('T1 (Pre-event)')
    axes[0].axis('off')
    
    axes[1].imshow(t2)
    axes[1].set_title('T2 (Post-event)')
    axes[1].axis('off')
    
    axes[2].imshow(mask, cmap='gray')
    axes[2].set_title('Change Mask')
    axes[2].axis('off')
    
    # Overlay
    overlay = t2.copy()
    overlay[mask > 0] = [255, 0, 0]
    blended = cv2.addWeighted(t2, 0.6, overlay, 0.4, 0)
    axes[3].imshow(blended)
    axes[3].set_title('T2 + Değişim Overlay')
    axes[3].axis('off')
    
    plt.suptitle(f'Örnek: {name}', fontsize=14, weight='bold')
    plt.show()